# Smart MRF: YOLO11 Waste Detection Training

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `data.zip` to `/content` (Files panel on the left)
3. Run the cells top to bottom

## 1. Install and mount Drive

In [ ]:
!pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

## 2. Check the GPU

In [ ]:
import torch
!nvidia-smi
assert torch.cuda.is_available(), "No GPU - go to Runtime > Change runtime type > T4 GPU, then re-run from the top"
print("GPU available:", torch.cuda.get_device_name(0))

## 3. Config

In [ ]:
from pathlib import Path

ZIP_PATH = Path("/content/data.zip")
RAW      = Path("/content/raw")
OUT      = Path("/content/combined")
RUN_DIR  = "/content/drive/MyDrive/smart-mrf-runs"   # saved on Drive so a reset can't lose it
RUN_NAME = "v1"
MODEL    = "yolo11s.pt"    # use yolo11n.pt for a faster test run
EPOCHS   = 120
BATCH    = 16

## 4. Unzip the data

In [ ]:
import shutil, zipfile

assert ZIP_PATH.exists(), f"Upload data.zip to /content first ({ZIP_PATH} not found)"
if RAW.exists(): shutil.rmtree(RAW)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(RAW)
if not (RAW/"Ds1").exists() and (RAW/"raw"/"Ds1").exists():
    RAW = RAW/"raw"
print("Found:", [p.name for p in RAW.iterdir()])

## 5. Check class names in each dataset

In [ ]:
import yaml

for ds in ["Ds1", "Ds2", "Ds3"]:
    names = yaml.safe_load(open(RAW/ds/"data.yaml"))["names"]
    print(ds, names)

## 6. Combine datasets into 8 classes
Check the printed mapping. If a class shows `-> None`, it is being dropped and needs a new rule.

In [ ]:
import re, shutil, yaml

if OUT.exists(): shutil.rmtree(OUT)

TARGET = ["paper","cardboard","plastic","plastic_bag","glass","metal","battery","aerosol"]
tid = {n: i for i, n in enumerate(TARGET)}

# first matching rule wins, so the order matters
RULES = [
    (r"aerosol", "aerosol"),
    (r"batter", "battery"),
    (r"paper bag", "paper"),
    (r"plastic bag|\bbag\b", "plastic_bag"),
    (r"glass", "glass"),
    (r"cardboard|carton", "cardboard"),
    (r"paper", "paper"),
    (r"metal|aluminu?m|\bcans?\b|\btin\b|steel", "metal"),
    (r"plastic|bottle|\bpet\b", "plastic"),
]
def target_of(name):
    n = re.sub(r"[_\-]+", " ", name.lower()).strip()
    for pat, t in RULES:
        if re.search(pat, n): return t
    return None

kept = {n: 0 for n in TARGET}
for ds in ["Ds1", "Ds2", "Ds3"]:
    names = yaml.safe_load(open(RAW/ds/"data.yaml"))["names"]
    if isinstance(names, dict): names = [names[i] for i in sorted(names)]
    old2new = {}
    for i, n in enumerate(names):
        t = target_of(n)
        print(f"{ds}: {n!r} -> {t}")
        if t: old2new[i] = tid[t]
    for split in ["train", "valid", "test"]:
        imgs = RAW/ds/split/"images"
        if not imgs.exists(): continue
        (OUT/split/"images").mkdir(parents=True, exist_ok=True)
        (OUT/split/"labels").mkdir(parents=True, exist_ok=True)
        for img in imgs.iterdir():
            lbl = RAW/ds/split/"labels"/(img.stem + ".txt")
            lines = []
            if lbl.exists():
                for ln in lbl.read_text().splitlines():
                    p = ln.split()
                    if len(p) == 5 and int(p[0]) in old2new:   # boxes only
                        lines.append(" ".join([str(old2new[int(p[0])])] + p[1:]))
                        kept[TARGET[old2new[int(p[0])]]] += 1
            if not lines: continue
            shutil.copy(img, OUT/split/"images"/f"{ds}_{img.name}")
            (OUT/split/"labels"/f"{ds}_{img.stem}.txt").write_text("\n".join(lines))

DATA = OUT/"data.yaml"
yaml.dump({"path": str(OUT), "train": "train/images", "val": "valid/images",
           "test": "test/images", "names": TARGET}, open(DATA, "w"))
print("\nBoxes kept per class:", kept)
assert DATA.exists()

## 7. Train

In [ ]:
from ultralytics import YOLO

assert DATA.exists(), "Run the combine cell first (the runtime was probably reset)"

model = YOLO(MODEL)
model.train(
    data=str(DATA),
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    patience=20,
    device=0,
    project=RUN_DIR,
    name=RUN_NAME,
    exist_ok=True,
)

## 8. Package and download

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

run  = Path(RUN_DIR)/RUN_NAME
best = run/"weights"/"best.pt"
assert best.exists(), f"Training didn't produce {best}"

shutil.copy(best, "/content/CV_model.pt")
shutil.make_archive("/content/CV_model", "zip", root_dir=run)   # whole run folder: weights, plots, results
print("Saved:", best)
files.download("/content/CV_model.zip")

## 9. (Optional) Preview predictions

In [ ]:
import glob
from IPython.display import Image, display
from ultralytics import YOLO

model = YOLO(f"{RUN_DIR}/{RUN_NAME}/weights/best.pt")
model.predict(source="/content/combined/valid/images", save=True)

for p in sorted(glob.glob('/content/runs/detect/predict*/*.jpg'))[:10]:
    display(Image(filename=p, height=400))